Save base Logit, KNN, SVC, LogisticRegression, CatBoost, XGboost with pipeline. Sedimentation_Stk feature was used!

Based on `research_14_SMOTE_influence_on_metrics.ipynb` it was decided to use SMOTE for all models, except Logit!

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

import optuna
from functools import partial

from pathlib import Path
import sys

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    _create_pipeline,
    StatsModelsEstimator,
    OptunaOptimizer,
    smote_objective,
    pure_smote_objective,
    GridSearchOptimizer,
    RANDOM_STATE,
)

seed = RANDOM_STATE

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'grid-opt-mean-smote_default-model'
add_smote = True
is_smotenc = True
smote_params = {
    'categorical_features': (
        'wettability',
        'inclination',
    ),
}
scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
n_trials = 50 # test only

save_model_and_metrics = True
metrics_file = 'metrics_modelling4_smotenc_mean_opt.xlsx'

## Optimization functions
Optuna will not be used further, except first try, it is enough to conduct grid search

In [5]:
def optimize_smote_optuna(
    target:str,
    estimator:object,
    estimator_params:dict,
    n_trials:int,
    step_scoring_average:str=step_scoring_average,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=save_model_and_metrics,
    metrics_file:str=metrics_file,
    seed:int=seed,
):
    """
    Optimize the SMOTE parameters for a given estimator.

    Args:
        target: The target variable for the model.
        estimator: The machine learning estimator to be optimized.
        estimator_params: Parameters for the estimator.
        n_trials: The number of trials for optimization. Defaults to n_trials.
        model_postfix: A postfix to append to the model name. Defaults to model_postfix.
        add_smote: Whether to add SMOTE to the pipeline. Defaults to add_smote.
        is_smotenc: Whether to use SMOTENC for categorical features. Defaults to is_smotenc.
        smote_params: Parameters for SMOTE. Defaults to smote_params.
        scoring_metrics: Metrics for scoring the model. Defaults to scoring_metrics.
        save_model_and_metrics: Whether to save the model and metrics. Defaults to save_model_and_metrics.
    """
    
    smote_params = smote_params.copy()
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics,
        step_scoring_average=step_scoring_average,
    )

    opt = OptunaOptimizer(
        objective=partial(smote_objective, ml_pipe=ml_pipe),
        study_name="smote_study",
        direction="maximize",
        seed=seed,
    )

    opt.optimize(n_trials=n_trials)

    best_params = opt.study.best_params
    print('best_params')
    display(best_params)

    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params={**smote_params, **best_params},
        metrics_file=metrics_file,
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )
    
    
def optimize_smote_grid(
    target:str,
    estimator:object,
    estimator_params:dict,
    param_grid:dict=None,
    step_scoring_average:str=step_scoring_average,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=save_model_and_metrics,
    metrics_file:str=metrics_file,
):
    """
    Optimize the SMOTE parameters with GridSearchOptimizer for a given estimator.

    Args:
        target: The target variable for the model.
        estimator: The machine learning estimator to be optimized.
        estimator_params: Parameters for the estimator.
        model_postfix: A postfix to append to the model name. Defaults to model_postfix.
        add_smote: Whether to add SMOTE to the pipeline. Defaults to add_smote.
        is_smotenc: Whether to use SMOTENC for categorical features. Defaults to is_smotenc.
        smote_params: Parameters for SMOTE. Defaults to smote_params.
        scoring_metrics: Metrics for scoring the model. Defaults to scoring_metrics.
        save_model_and_metrics: Whether to save the model and metrics. Defaults to save_model_and_metrics.
    """
    smote_params = smote_params.copy()
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics,
        step_scoring_average=step_scoring_average,
    )
    
    if param_grid is None:
        param_grid = {
        'sampling_strategy': [0.6, 0.7, 0.8, 0.9, 1.0, 'auto'],
        'k_neighbors': list(range(2,11)),
    }
    opt = GridSearchOptimizer(
        objective=partial(pure_smote_objective, ml_pipe=ml_pipe),
        param_grid=param_grid,
        verbose=True,
    )

    opt.optimize()
    best_params = opt.best_params

    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params={**smote_params, **best_params},
        metrics_file=metrics_file,
        
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

# Logistic Regression

In [6]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    n_trials=n_trials,
    save_model_and_metrics=False,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-15 22:29:26,379] A new study created in memory with name: smote_study
[I 2025-04-15 22:29:26,743] Trial 0 finished with value: 0.799204684219129 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.799204684219129.
[I 2025-04-15 22:29:26,859] Trial 1 finished with value: 0.8082274269002464 and parameters: {'k_neighbors': 3, 'sampling_strategy': 1.0}. Best is trial 1 with value: 0.8082274269002464.
[I 2025-04-15 22:29:26,970] Trial 2 finished with value: 0.7997594024534191 and parameters: {'k_neighbors': 9, 'sampling_strategy': 1.0}. Best is trial 1 with value: 0.8082274269002464.
[I 2025-04-15 22:29:27,072] Trial 3 finished with value: 0.8082591916442389 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.7}. Best is trial 3 with value: 0.8082591916442389.
[I 2025-04-15 22:29:27,171] Trial 4 finished with value: 0.8142513697328472 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.6}. Best is trial 4 with value: 0.814251369

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smotenc_grid-opt-...
holdout_test_f1_macro,0.793956
holdout_test_accuracy_balanced,0.789352
holdout_test_roc_auc,0.875
holdout_test_f1,0.857143
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.799242
cv_test_accuracy_balanced_median,0.816667
cv_test_roc_auc_median,0.876161


In [7]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   4%|▎         | 2/54 [00:00<00:10,  5.00it/s]

Progress: 1/54.	Score: 0.8065845885618185.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.
Progress: 2/54.	Score: 0.8091164906318441.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.7}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smotenc_grid-opt-...
holdout_test_f1_macro,0.793956
holdout_test_accuracy_balanced,0.789352
holdout_test_roc_auc,0.875
holdout_test_f1,0.857143
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.799242
cv_test_accuracy_balanced_median,0.816667
cv_test_roc_auc_median,0.876161


Model saved in ../results/models_modelling4/LogisticRegression_splashing_smotenc_grid-opt-mean-smote_default-model


Full grid search can find better solution, than Optuna with small number of iterations - like 40! It appears in some Random_seed

In [8]:
estimator = LogisticRegression
target = 'no_fragmentation'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:05,  9.94it/s]

Progress: 1/54.	Score: 0.8028993359522488.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LogisticRegression"


,0
target,no_fragmentation
model,LogisticRegression_no_fragmentation_smotenc_gr...
holdout_test_f1_macro,0.798595
holdout_test_accuracy_balanced,0.834091
holdout_test_roc_auc,0.95
holdout_test_f1,0.723404
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.826797
cv_test_accuracy_balanced_median,0.869231
cv_test_roc_auc_median,0.946886


Model saved in ../results/models_modelling4/LogisticRegression_no_fragmentation_smotenc_grid-opt-mean-smote_default-model


# KNClassifier

In [9]:
estimator = KNeighborsClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:07,  6.77it/s]

Progress: 1/54.	Score: 0.8306449960120383.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "KNeighborsClassifier"


,0
target,splashing
model,KNeighborsClassifier_splashing_smotenc_grid-op...
holdout_test_f1_macro,0.813397
holdout_test_accuracy_balanced,0.815972
holdout_test_roc_auc,0.880015
holdout_test_f1,0.863158
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.844575
cv_test_accuracy_balanced_median,0.870743
cv_test_roc_auc_median,0.916409


Model saved in ../results/models_modelling4/KNeighborsClassifier_splashing_smotenc_grid-opt-mean-smote_default-model


In [10]:
estimator = KNeighborsClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:09,  5.75it/s]

Progress: 1/54.	Score: 0.8199108903889971.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "KNeighborsClassifier"


,0
target,no_fragmentation
model,KNeighborsClassifier_no_fragmentation_smotenc_...
holdout_test_f1_macro,0.834656
holdout_test_accuracy_balanced,0.845455
holdout_test_roc_auc,0.947727
holdout_test_f1,0.761905
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.854396
cv_test_accuracy_balanced_median,0.854396
cv_test_roc_auc_median,0.931319


Model saved in ../results/models_modelling4/KNeighborsClassifier_no_fragmentation_smotenc_grid-opt-mean-smote_default-model


# SVC

In [11]:
estimator = SVC
target = 'splashing'
estimator_params = {
    'probability': True,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:07,  7.48it/s]

Progress: 1/54.	Score: 0.8520810728898827.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "SVC"


,0
target,splashing
model,SVC_splashing_smotenc_grid-opt-mean-smote_defa...
holdout_test_f1_macro,0.810348
holdout_test_accuracy_balanced,0.80787
holdout_test_roc_auc,0.900463
holdout_test_f1,0.865979
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.860788
cv_test_accuracy_balanced_median,0.873839
cv_test_roc_auc_median,0.928793


Model saved in ../results/models_modelling4/SVC_splashing_smotenc_grid-opt-mean-smote_default-model


In [12]:
estimator = SVC
target = 'no_fragmentation'
estimator_params = {
    'probability': True,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:06,  7.61it/s]

Progress: 1/54.	Score: 0.8162029215024607.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "SVC"


,0
target,no_fragmentation
model,SVC_no_fragmentation_smotenc_grid-opt-mean-smo...
holdout_test_f1_macro,0.825397
holdout_test_accuracy_balanced,0.852273
holdout_test_roc_auc,0.951818
holdout_test_f1,0.755556
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.850704
cv_test_accuracy_balanced_median,0.89011
cv_test_roc_auc_median,0.957875


Model saved in ../results/models_modelling4/SVC_no_fragmentation_smotenc_grid-opt-mean-smote_default-model


# CatBoost

In [13]:
estimator = CatBoostClassifier
target = 'splashing'
estimator_params = {
    'verbose': False,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:05<04:33,  5.16s/it]

Progress: 1/54.	Score: 0.8822828366606001.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "CatBoostClassifier"


,0
target,splashing
model,CatBoostClassifier_splashing_smotenc_grid-opt-...
holdout_test_f1_macro,0.882261
holdout_test_accuracy_balanced,0.876157
holdout_test_roc_auc,0.956019
holdout_test_f1,0.918367
holdout_test_accuracy,0.893333
cv_test_f1_macro_median,0.876935
cv_test_accuracy_balanced_median,0.876935
cv_test_roc_auc_median,0.956656


Model saved in ../results/models_modelling4/CatBoostClassifier_splashing_smotenc_grid-opt-mean-smote_default-model


In [14]:
estimator = CatBoostClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': False,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:05<04:26,  5.03s/it]

Progress: 1/54.	Score: 0.8445879172947282.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "CatBoostClassifier"


,0
target,no_fragmentation
model,CatBoostClassifier_no_fragmentation_smotenc_gr...
holdout_test_f1_macro,0.897727
holdout_test_accuracy_balanced,0.897727
holdout_test_roc_auc,0.979091
holdout_test_f1,0.85
holdout_test_accuracy,0.92
cv_test_f1_macro_median,0.898077
cv_test_accuracy_balanced_median,0.880037
cv_test_roc_auc_median,0.97094


Model saved in ../results/models_modelling4/CatBoostClassifier_no_fragmentation_smotenc_grid-opt-mean-smote_default-model


# XGBoost

In [15]:
estimator = XGBClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:01<01:29,  1.68s/it]

Progress: 1/54.	Score: 0.8889781084579695.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "XGBClassifier"


,0
target,splashing
model,XGBClassifier_splashing_smotenc_grid-opt-mean-...
holdout_test_f1_macro,0.846814
holdout_test_accuracy_balanced,0.831019
holdout_test_roc_auc,0.942901
holdout_test_f1,0.901961
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.879545
cv_test_accuracy_balanced_median,0.888545
cv_test_roc_auc_median,0.947619


Model saved in ../results/models_modelling4/XGBClassifier_splashing_smotenc_grid-opt-mean-smote_default-model


In [16]:
estimator = XGBClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:01<01:25,  1.62s/it]

Progress: 1/54.	Score: 0.8352164170195232.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "XGBClassifier"


,0
target,no_fragmentation
model,XGBClassifier_no_fragmentation_smotenc_grid-op...
holdout_test_f1_macro,0.885894
holdout_test_accuracy_balanced,0.904545
holdout_test_roc_auc,0.988182
holdout_test_f1,0.837209
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.877289
cv_test_roc_auc_median,0.967033


Model saved in ../results/models_modelling4/XGBClassifier_no_fragmentation_smotenc_grid-opt-mean-smote_default-model


# AdaBoost

In [17]:
estimator = AdaBoostClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:19,  2.74it/s]

Progress: 1/54.	Score: 0.8330454887299584.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "AdaBoostClassifier"


,0
target,splashing
model,AdaBoostClassifier_splashing_smotenc_grid-opt-...
holdout_test_f1_macro,0.743541
holdout_test_accuracy_balanced,0.747685
holdout_test_roc_auc,0.873457
holdout_test_f1,0.808511
holdout_test_accuracy,0.76
cv_test_f1_macro_median,0.789683
cv_test_accuracy_balanced_median,0.80031
cv_test_roc_auc_median,0.877709


Model saved in ../results/models_modelling4/AdaBoostClassifier_splashing_smotenc_grid-opt-mean-smote_default-model


In [18]:
estimator = AdaBoostClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:17,  3.04it/s]

Progress: 1/54.	Score: 0.8050520087762468.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "AdaBoostClassifier"


,0
target,no_fragmentation
model,AdaBoostClassifier_no_fragmentation_smotenc_gr...
holdout_test_f1_macro,0.825397
holdout_test_accuracy_balanced,0.852273
holdout_test_roc_auc,0.938182
holdout_test_f1,0.755556
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.826067
cv_test_accuracy_balanced_median,0.818681
cv_test_roc_auc_median,0.923077


Model saved in ../results/models_modelling4/AdaBoostClassifier_no_fragmentation_smotenc_grid-opt-mean-smote_default-model


# Random Forest

In [19]:
estimator = RandomForestClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:30,  1.76it/s]

Progress: 1/54.	Score: 0.8725994948933762.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "RandomForestClassifier"


,0
target,splashing
model,RandomForestClassifier_splashing_smotenc_grid-...
holdout_test_f1_macro,0.816176
holdout_test_accuracy_balanced,0.802083
holdout_test_roc_auc,0.951775
holdout_test_f1,0.882353
holdout_test_accuracy,0.84
cv_test_f1_macro_median,0.896201
cv_test_accuracy_balanced_median,0.891641
cv_test_roc_auc_median,0.94582


Model saved in ../results/models_modelling4/RandomForestClassifier_splashing_smotenc_grid-opt-mean-smote_default-model


In [20]:
estimator = RandomForestClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:29,  1.81it/s]

Progress: 1/54.	Score: 0.8235176629197435.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "RandomForestClassifier"


,0
target,no_fragmentation
model,RandomForestClassifier_no_fragmentation_smoten...
holdout_test_f1_macro,0.867725
holdout_test_accuracy_balanced,0.879545
holdout_test_roc_auc,0.977727
holdout_test_f1,0.809524
holdout_test_accuracy,0.893333
cv_test_f1_macro_median,0.898077
cv_test_accuracy_balanced_median,0.880037
cv_test_roc_auc_median,0.94359


Model saved in ../results/models_modelling4/RandomForestClassifier_no_fragmentation_smotenc_grid-opt-mean-smote_default-model


# LightGBM

In [21]:
estimator = LGBMClassifier
target = 'splashing'
estimator_params = {
    'verbose': -1,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:02<02:08,  2.42s/it]

Progress: 1/54.	Score: 0.8664370418236186.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LGBMClassifier"


,0
target,splashing
model,LGBMClassifier_splashing_smotenc_grid-opt-mean...
holdout_test_f1_macro,0.882261
holdout_test_accuracy_balanced,0.876157
holdout_test_roc_auc,0.954861
holdout_test_f1,0.918367
holdout_test_accuracy,0.893333
cv_test_f1_macro_median,0.881696
cv_test_accuracy_balanced_median,0.900155
cv_test_roc_auc_median,0.952012


Model saved in ../results/models_modelling4/LGBMClassifier_splashing_smotenc_grid-opt-mean-smote_default-model


In [22]:
estimator = LGBMClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': -1,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:02<02:16,  2.57s/it]

Progress: 1/54.	Score: 0.8197538039317319.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LGBMClassifier"


,0
target,no_fragmentation
model,LGBMClassifier_no_fragmentation_smotenc_grid-o...
holdout_test_f1_macro,0.933862
holdout_test_accuracy_balanced,0.947727
holdout_test_roc_auc,0.989091
holdout_test_f1,0.904762
holdout_test_accuracy,0.946667
cv_test_f1_macro_median,0.876543
cv_test_accuracy_balanced_median,0.867216
cv_test_roc_auc_median,0.964103


Model saved in ../results/models_modelling4/LGBMClassifier_no_fragmentation_smotenc_grid-opt-mean-smote_default-model


# For the notebook with Model optimization!

In [23]:
# def update_estimator_params(
#     ml_pipe:MLPipeline,
#     suggested_params:dict,
# ) -> dict:
#     """Update the estimator parameters based on the pipeline parameters.

#     Args:
#         ml_pipe: An instance of MLPipeline used for model training and evaluation.
#         suggested_params: A dictionary containing the suggested Estimator hyperparameters.
    
#     Returns:
#         A dictionary containing the estimator parameters.
#     """
#     estimator_params = ml_pipe._pipeline_params['estimator_params']
#     estimator_params.update(suggested_params)
#     return estimator_params

# def logreg_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
#     """Objective function for logistic regression optimization using Optuna.

#     Args:
#         trial: An Optuna trial object used to suggest hyperparameters.
#         ml_pipe: An instance of MLPipeline used for model training and evaluation.

#     Returns:
#         The score of the model based on the specified evaluation metric.
#     """
    
#     categorical_features = ('wettability', 'inclination')
    
#     random_state = ml_pipe._pipeline_params['random_state']
    
#     # sample params
#     C = trial.suggest_loguniform('C', 1e-4, 1e3)
    
#     # SMOTE params
#     add_smote = trial.suggest_categorical('add_smote', [True, False])
#     if add_smote:
#         is_smotenc = trial.suggest_categorical('is_smotenc', [True, False])
#         smote_params = {
#             'sampling_strategy': trial.suggest_categorical(
#                 'sampling_strategy', [0.6, 0.7, 0.8, 0.9, 1.0]
#             ),
#             'k_neighbors': trial.suggest_int('k_neighbors', 3, 10),
#             'random_state': random_state,
#         }
#     else:
#         is_smotenc = False
#         smote_params = None
#     if is_smotenc:
#         wettability_cat = trial.suggest_categorical('wettability_cat', [True, False])
#         if wettability_cat:
#             inclination_cat = trial.suggest_categorical('inclination_cat', [True, False])
#         else:
#             inclination_cat = True
        
        
#         mask = [wettability_cat, inclination_cat]
        
#         masked_features = [
#             feature for feature, mask_value in zip(categorical_features, mask) 
#             if mask_value
#         ]
#         smote_params = {
#             **smote_params,
#             'categorical_features': masked_features,
#         }

#     suggested_params = {
#         "C": C,
#     }
    
#     estimator_params = update_estimator_params(ml_pipe, suggested_params)

#     estimator = LogisticRegression(
#         **estimator_params,
#         random_state=random_state,
#     )

#     score = ml_pipe.step(
#         estimator=estimator,
#         add_smote=add_smote,
#         is_smotenc=is_smotenc,
#         smote_params=smote_params,
#     )
    
#     return score



# opt = OptunaOptimizer(
#     objective=partial(logreg_objective, ml_pipe=ml_pipe),
#     study_name="logreg_study",
#     direction="maximize",
# )

# opt.optimize(n_trials=200)

# best_params = opt.study.best_trial.params
# display(best_params)
# # estimator_params = update_estimator_params(ml_pipe, best_params)

# # estimator = LogisticRegression(
# #     **estimator_params,
# #     random_state=ml_pipe._pipeline_params['random_state'],
# # )